In [47]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import MNIST

In [48]:
# Dataset & DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = MNIST(root="./data", train=True, download=True, transform=transform)
testset = MNIST(root="./data", train=False, download=True, transform=transform)

In [49]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build the CNN

In [50]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128 , 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x
        

In [51]:
model = CNN()

In [52]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Train CNN Model

In [53]:
# training
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()

    # validation        
    model.eval()
    validation_loss = 0.0

    with torch.no_grad():
        for images, labels in testloader:
            outputs = model.forward(images)
            loss = criterion(outputs, labels)

            validation_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} ==> loss = {epoch_train_loss / len(trainloader)} && validation loss = {validation_loss / len(testloader)}")

# model save
torch.save(model.state_dict(), "mnist_cnn.pth")
print("Model saved successfully")

epoch = 1/10 ==> loss = 0.14777602766976078 && validation loss = 0.04311455133783328
epoch = 2/10 ==> loss = 0.040701496834503186 && validation loss = 0.03941344447976355
epoch = 3/10 ==> loss = 0.028877073964891534 && validation loss = 0.03658534910885169
epoch = 4/10 ==> loss = 0.024063112603510167 && validation loss = 0.03080791420043575
epoch = 5/10 ==> loss = 0.016647176422272883 && validation loss = 0.030526579761945363
epoch = 6/10 ==> loss = 0.015115019382155762 && validation loss = 0.031510804548196096
epoch = 7/10 ==> loss = 0.012777174884005657 && validation loss = 0.031613161009727844
epoch = 8/10 ==> loss = 0.012399804863149773 && validation loss = 0.024937375461740956
epoch = 9/10 ==> loss = 0.008515441259966682 && validation loss = 0.03206612038114153
epoch = 10/10 ==> loss = 0.009350230604627396 && validation loss = 0.029503779320913536
Model saved successfully


In [25]:
# Evaluate
correct_labels = 0.0
total_labels = 0.0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 99.29


# Build our RNN

In [43]:
class RNN(nn.Module):
    def __init__(self, input_size=28, hidden_state=128, num_layers=2):
        super().__init__()

        self.hidden_state = hidden_state
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_state, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_state, 10)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_state)

        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])

        return out

In [44]:
model = RNN()

## Train the RNN

In [45]:
epochs = 50

for epoch in range(epochs):
    model.train()

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        images = images.squeeze(1)
        
        outputs = model(images)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} ==> loss = {loss.item()}")        

epoch = 1/50 ==> loss = 2.3229775428771973
epoch = 2/50 ==> loss = 2.31038236618042
epoch = 3/50 ==> loss = 2.315575122833252
epoch = 4/50 ==> loss = 2.327939033508301
epoch = 5/50 ==> loss = 2.3130528926849365
epoch = 6/50 ==> loss = 2.2794864177703857
epoch = 7/50 ==> loss = 2.311141014099121
epoch = 8/50 ==> loss = 2.330246686935425
epoch = 9/50 ==> loss = 2.2914586067199707
epoch = 10/50 ==> loss = 2.339367389678955
epoch = 11/50 ==> loss = 2.3198153972625732
epoch = 12/50 ==> loss = 2.3172247409820557
epoch = 13/50 ==> loss = 2.323197841644287
epoch = 14/50 ==> loss = 2.2821598052978516
epoch = 15/50 ==> loss = 2.3215479850769043
epoch = 16/50 ==> loss = 2.3168387413024902
epoch = 17/50 ==> loss = 2.367924213409424
epoch = 18/50 ==> loss = 2.3264193534851074
epoch = 19/50 ==> loss = 2.344900608062744
epoch = 20/50 ==> loss = 2.3247804641723633
epoch = 21/50 ==> loss = 2.3350818157196045
epoch = 22/50 ==> loss = 2.359619379043579
epoch = 23/50 ==> loss = 2.310079336166382
epoch = 2

In [46]:
# Evaluate 
model.eval()

correct_labels = 0.0
total_labels = 0.0

with torch.no_grad():
    for images, labels in testloader:

        images = images.squeeze(1)

        outputs = model(images)

        _,predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
        
print(f"accuracy = {correct_labels/total_labels * 100}")        

accuracy = 8.92
